# Purpose: Find out why some mutants are different than others.

In [ ]:
from scipy.io import loadmat
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
from mapd import Trial, Table, Sinq 
from mapd.sinq_builders import build_composite_sinq, subset_sinq
import h5py
import glob
import numpy as np
import pickle
import plotly.express as px
import matplotlib.colors as mcolors
# import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
import matplotlib as mpl
mpl.rcParams.update(mpl.rcParamsDefault)  # reset to defaults
mpl.rcParams['pdf.fonttype'] = 42         # embed fonts as text, not paths
mpl.rcParams['svg.fonttype'] = 'none'     # keep text editable in SVG
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['font.size'] = 11

import seaborn as sns

import random

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

from IPython.display import display

%load_ext autoreload
%autoreload 2


%matplotlib widget

def refresh_table_addons():
    import importlib
    import mapd.table_plotters as tps
    import mapd.table_scalars as tbs
    importlib.reload(tps)
    importlib.reload(tbs)

    for name in dir(tps):
        if name.startswith('_'):
            continue
        attr = getattr(tps, name)
        setattr(Table, name[len("plot_"):], attr)
    
    for name in dir(tbs):
        if name.startswith('_'):
            continue
        if name.startswith('compute_'):
            attr = getattr(tbs, name)
            setattr(Table, name[len("compute_"):], attr)


# Fig dir for PCA analysis
mutant_fig_dir = 'Figure5_mutant plots'

os.makedirs(mutant_fig_dir, exist_ok=True)

# Load learners vs. control sinq

In [ ]:
sinq_1_2 = Sinq(sinqname='dataset_1_2_pca_lda') # Constructed in Figure4_Fig5 code

# Import mutant sinq and select other rows

In [ ]:
sinq_g = Sinq(sinqname='dataset_3_genetic_manipulations')
print(sinq_g.__repr__())

# Per-state probe position across trials
Test of new `Table.plot_state_positions` — shows where the probe sits during REST, DRIFT, and MOVE samples on each trial, with target bands for context. Useful for asking whether some flies hold the probe beyond the target zone at rest.

Compare a couple of lines here. Might think about looping back and including these metrics in the pca? 

One argument for not including the drift movement rest thing in pca/lda is that so far it's just to quantify one thing. Measuring additional parameters on top of this is pserhaps a bit of a stretch

In [ ]:
refresh_table_addons()

# Pick a couple of flies from the loaded sinq to inspect.
# Use the same flies already referenced in this notebook plus a learner.
# test_dfcs = {'251107_F2_C1', '251103_F2_C1'}
test_dfcs = sinq_1_2.display_df()[sinq_1_2.has_tag('1s_stim')].index

for dfc in test_dfcs:
    fig = Figure(figsize=(11, 4), dpi=150)
    FigureCanvas(fig)
    ax = fig.add_subplot(111)

    T = sinq_1_2.restore_table(dfc)
    T.extract_trial_properties()
    
    T.plot_state_positions(percentiles=(50,),states=('rest',), ax=ax)
    display(fig)

# PPL state positions

In [ ]:
ppl_df = sinq_g.df.loc[sinq_g.has_tag('PPL')]

In [ ]:
for dfc in ppl_df.index:
    fig = Figure(figsize=(11, 4), dpi=150)
    FigureCanvas(fig)
    ax = fig.add_subplot(111)

    T = sinq_g.restore_table(dfc)
    T.extract_trial_properties()
    
    T.plot_state_positions(percentiles=(50,),states=('rest',), ax=ax)
    display(fig)

## Compare the PPL1 to white mutants that also fail

## Compare PPL1 to white and iav mutants that really suck.

# IAV explanation

Trial 492 of 251217_F2_C1, shows drift while trying to find target
Trial 252-268 of 250320_F1_C1, shows drift while trying to find target

In [ ]:
iav_df = sinq_g.df.loc[sinq_g.has_tag('+') & sinq_g.has_tag('iav')]

for dfc in iav_df.index:
    fig = Figure(figsize=(11, 4), dpi=150)
    FigureCanvas(fig)
    ax = fig.add_subplot(111)

    T = sinq_g.restore_table(dfc)
    T.extract_trial_properties()
    
    T.plot_state_positions(percentiles=(50,),states=('rest',), ax=ax)
    sinq_g.drop_tables()
    display(fig)